In [0]:
%run /Workspace/weather_notebook/nb_utils_dev

In [0]:
# ── Cell 2: Read Bronze ───────────────────────────────────────
log_header("weather_readings")
print("\n[1/4] Reading Bronze JSON...")

json_path = list_bronze_json_paths("weather/current")
df_raw    = spark.read.option("multiline", "true").json(json_path)
print(f"  Raw rows: {df_raw.count():,}")

In [0]:
# ── Cell 3: Transform ─────────────────────────────────────────
print("\n[2/4] Transforming...")

# Handle optional columns before main chain
if "rain" not in df_raw.columns:
    df_raw = df_raw.withColumn("rain", F.lit(None).cast("string"))
if "snow" not in df_raw.columns:
    df_raw = df_raw.withColumn("snow", F.lit(None).cast("string"))

# Step 1: Basic column extractions
df1 = df_raw \
    .withColumn("city_name",           F.col("name")) \
    .withColumn("country",             F.col("sys.country")) \
    .withColumn("latitude",            F.col("coord.lat").cast("double")) \
    .withColumn("longitude",           F.col("coord.lon").cast("double")) \
    .withColumn("weather_id",          F.col("weather")[0]["id"].cast("integer")) \
    .withColumn("weather_main",        F.col("weather")[0]["main"]) \
    .withColumn("weather_description", F.col("weather")[0]["description"]) \
    .withColumn("weather_icon",        F.col("weather")[0]["icon"]) \
    .withColumn("temperature_c",       F.col("main.temp").cast("double")) \
    .withColumn("feels_like_c",        F.col("main.feels_like").cast("double")) \
    .withColumn("temp_min_c",          F.col("main.temp_min").cast("double")) \
    .withColumn("temp_max_c",          F.col("main.temp_max").cast("double")) \
    .withColumn("pressure_hpa",        F.col("main.pressure").cast("integer")) \
    .withColumn("humidity_pct",        F.col("main.humidity").cast("integer")) \
    .withColumn("wind_speed_ms",       F.col("wind.speed").cast("double")) \
    .withColumn("wind_direction_deg",  F.col("wind.deg").cast("integer")) \
    .withColumn("wind_gust_ms",
        F.when(F.col("wind.gust").isNotNull(), F.col("wind.gust").cast("double")).otherwise(F.lit(0.0))) \
    .withColumn("visibility_m",
        F.when(F.col("visibility").isNotNull(), F.col("visibility").cast("integer")).otherwise(F.lit(10000))) \
    .withColumn("cloud_cover_pct",     F.col("clouds.all").cast("integer")) \
    .withColumn("rain_1h_mm",          F.lit(0.0).cast("double")) \
    .withColumn("snow_1h_mm",          F.lit(0.0).cast("double")) \
    .withColumn("reading_timestamp",   F.from_unixtime(F.col("dt")).cast("timestamp")) \
    .withColumn("sunrise_timestamp",   F.from_unixtime(F.col("sys.sunrise")).cast("timestamp")) \
    .withColumn("sunset_timestamp",    F.from_unixtime(F.col("sys.sunset")).cast("timestamp")) \
    .withColumn("reading_date",        F.to_date(F.col("reading_timestamp"))) \
    .withColumn("reading_hour",        F.hour(F.col("reading_timestamp")))

# Step 2: Derived columns
df2 = df1 \
    .withColumn("temp_range_c",
        F.round(F.col("temp_max_c") - F.col("temp_min_c"), 2)) \
    .withColumn("is_daytime",
        F.when(
            (F.unix_timestamp("reading_timestamp") > F.unix_timestamp("sunrise_timestamp")) &
            (F.unix_timestamp("reading_timestamp") < F.unix_timestamp("sunset_timestamp")), 1
        ).otherwise(0)) \
    .withColumn("daylight_hours",
        F.round((F.unix_timestamp("sunset_timestamp") - F.unix_timestamp("sunrise_timestamp")) / 3600, 2)) \
    .withColumn("wind_beaufort",
        F.when(F.col("wind_speed_ms") < 0.3,  F.lit(0))
         .when(F.col("wind_speed_ms") < 1.6,  F.lit(1))
         .when(F.col("wind_speed_ms") < 3.4,  F.lit(2))
         .when(F.col("wind_speed_ms") < 5.5,  F.lit(3))
         .when(F.col("wind_speed_ms") < 8.0,  F.lit(4))
         .when(F.col("wind_speed_ms") < 10.8, F.lit(5))
         .when(F.col("wind_speed_ms") < 13.9, F.lit(6))
         .otherwise(F.lit(7)))

# Step 3: Categorical columns
df3 = df2 \
    .withColumn("weather_severity",
        F.when(F.col("weather_main").isin("Thunderstorm", "Tornado"), F.lit("severe"))
         .when(F.col("weather_main").isin("Snow", "Sleet", "Blizzard"), F.lit("winter"))
         .when(F.col("weather_main").isin("Rain", "Drizzle"), F.lit("wet"))
         .when(F.col("weather_main") == "Fog",   F.lit("poor_visibility"))
         .when(F.col("weather_main") == "Clear", F.lit("clear"))
         .otherwise(F.lit("normal"))) \
    .withColumn("comfort_level",
        F.when(
            (F.col("temperature_c").between(18, 24)) &
            (F.col("humidity_pct").between(40, 60)) &
            (F.col("wind_speed_ms") < 5), F.lit("comfortable"))
         .when(F.col("temperature_c") < 0,  F.lit("freezing"))
         .when(F.col("temperature_c") < 10, F.lit("cold"))
         .when(F.col("temperature_c") < 18, F.lit("cool"))
         .when(F.col("temperature_c") > 30, F.lit("hot"))
         .otherwise(F.lit("moderate"))) \
    .withColumn("ingestion_date",  F.current_date()) \
    .withColumn("ingestion_ts",    F.current_timestamp()) \
    .withColumn("source_system",   F.lit("openweathermap_api")) \
    .withColumn("source_endpoint", F.lit("/data/2.5/weather"))

# Step 4: Filter, deduplicate, drop raw columns, select final
silver_weather = df3 \
    .filter(F.col("city_name").isNotNull()) \
    .filter(F.col("temperature_c").between(-50, 60)) \
    .filter(F.col("humidity_pct").between(0, 100)) \
    .withColumn("_rn",
        F.row_number().over(
            Window.partitionBy("city_name", "reading_date", "reading_hour")
                  .orderBy(F.desc("reading_timestamp")))) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn", "coord", "weather", "main", "wind",
          "clouds", "sys", "dt", "base", "id",
          "timezone", "visibility", "name", "rain", "snow") \
    .select(
        "city_name", "country", "latitude", "longitude",
        "reading_timestamp", "reading_date", "reading_hour",
        "weather_id", "weather_main", "weather_description",
        "weather_icon", "weather_severity", "comfort_level",
        "temperature_c", "feels_like_c", "temp_min_c",
        "temp_max_c", "temp_range_c",
        "pressure_hpa", "humidity_pct",
        "wind_speed_ms", "wind_direction_deg",
        "wind_gust_ms", "wind_beaufort",
        "visibility_m", "cloud_cover_pct",
        "rain_1h_mm", "snow_1h_mm",
        "sunrise_timestamp", "sunset_timestamp",
        "is_daytime", "daylight_hours",
        "ingestion_date", "ingestion_ts",
        "source_system", "source_endpoint"
    )

row_count = silver_weather.count()
print(f"  Rows: {row_count:,}")

In [0]:
# ── Cell 4: Write to ADLS Gen2 + register in Unity Catalog ───
print("\n[3/4] Writing to ADLS Gen2...")
row_count = write_silver_table(silver_weather, "weather_readings")

In [0]:
# ── Cell 5: Verify + preview ──────────────────────────────────
print("\n[4/4] Verifying...")
total, nulls, dups, status = verify_unity_table("weather_readings", "city_name")

print("\nLatest readings per city:")
spark.sql(f"""
    SELECT city_name, reading_date, reading_hour,
           temperature_c, weather_main, humidity_pct,
           comfort_level, wind_beaufort
    FROM {silver_catalog}.weather_readings
    ORDER BY reading_timestamp DESC
    LIMIT 10
""").show(truncate=False)

print("\nComfort level distribution:")
spark.sql(f"""
    SELECT comfort_level, COUNT(*) as count,
           ROUND(AVG(temperature_c), 2) as avg_temp
    FROM {silver_catalog}.weather_readings
    GROUP BY comfort_level
    ORDER BY count DESC
""").show()

log_footer("weather_readings", total, status)
dbutils.notebook.exit(f"weather_readings|{total}|{status}")